In [1]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("smoke-test")

with mlflow.start_run(run_name="hello"):
    mlflow.log_param("model", "fake")
    mlflow.log_metric("spearman", 0.42)

print("logged")

c:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/08 09:34:27 INFO mlflow.tracking.fluent: Experiment with name 'smoke-test' does not exist. Creating a new experiment.


🏃 View run hello at: http://127.0.0.1:5000/#/experiments/2/runs/914cb10ad9cd4ed9b34cb436a0a2d0fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
logged


In [1]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\history\all_seasons_fixed.parquet")
d = df[df["season"] == "2025-26"]

print("GWs present:", sorted(d["round"].unique()))
print("Rows missing value:", d["value"].isna().sum())

# does price actually move? if it never changes, sell-price logic is moot
piv = d.groupby("element")["value"].agg(["min", "max", "nunique"])
print("\nPlayers whose price never moved:", (piv["nunique"] == 1).sum())
print("Players whose price moved:      ", (piv["nunique"] > 1).sum())
print("Biggest rise (tenths):", (piv["max"] - piv["min"]).max())

GWs present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38)]
Rows missing value: 0

Players whose price never moved: 241
Players whose price moved:       600
Biggest rise (tenths): 15


In [2]:
import pandas as pd
p = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\predictions_2526.parquet")
g1 = p[p["gw"] == 1]
print(g1[g1["name"].str.contains("Haaland|Salah", case=False)][["name","team","e_points","e_minutes"]])

                 name       team  e_points  e_minutes
14200   Mohamed Salah  Liverpool  7.963638  71.026191
16007  Erling Haaland   Man City  6.599158  73.052870


In [3]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\history\all_seasons_fixed.parquet")
d = df[df["season"] == "2025-26"]
print([c for c in d.columns if c in
  ["element","round","GW","minutes","total_points","position","team","value"]])
print(d[d["round"]==1][["element","minutes","total_points"]].head(3))

['element', 'minutes', 'round', 'total_points', 'value', 'GW', 'position', 'team']
        element  minutes  total_points
224143      541       90             6
224144       57        0             0
224145       87        0             0


In [1]:
import pandas as pd
w = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\walkforward_2526.parquet")
print(w.columns.tolist())
print(w.shape)
print(sorted(w["gw"].unique()))
print(w.head(3))

['element', 'gw', 'name', 'position', 'team', 'minutes', 'actual_points', 'player_id', 'understat_id', 'p_start', 'p60', 'e_minutes', 'understat_id_num', 'understat_id_r', 'npxg90', 'xa90', 'team_lambda', 'opp_lambda', 'p_cs', 'p_dc_hit', 'minutes_frac', 'fixture_scale', 'e_goals', 'e_assists', 'pts_goals', 'pts_assists', 'p_60plus', 'p_play_any', 'pts_appear', 'pts_cs', 'pts_dc', 'e_points_core', 'pred_bps', 'exp_bonus', 'e_points']
(29338, 35)
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38)]
   element  gw            

In [2]:
import pandas as pd
w = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\walkforward_2526.parquet")
p = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\predictions_2526.parquet")

for gw in [1, 2, 20, 38]:
    a = w[w["gw"]==gw].set_index("element")["e_points"]
    b = p[p["gw"]==gw].set_index("element")["e_points"]
    j = a.to_frame("wf").join(b.to_frame("static"), how="inner")
    print(f"GW{gw}: n={len(j)}  identical={j['wf'].equals(j['static'])}  "
          f"corr={j['wf'].corr(j['static']):.4f}  mean_abs_diff={(j['wf']-j['static']).abs().mean():.4f}")

GW1: n=690  identical=False  corr=1.0000  mean_abs_diff=0.0006
GW2: n=705  identical=False  corr=0.9989  mean_abs_diff=0.0420
GW20: n=790  identical=False  corr=0.9979  mean_abs_diff=0.0501
GW38: n=841  identical=False  corr=0.9950  mean_abs_diff=0.0566


In [4]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season

season = load_season()
state, log = simulate_season(season, gws=[1, 2, 3])

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7


In [5]:
log[["gw","points","raw_points","captain","captain_bonus","doubled_role","n_subs","bench_points"]]

,gw,points,raw_points,captain,captain_bonus,doubled_role,n_subs,bench_points
0,1,47,47,Mohamed Salah,8,captain,1,3
1,2,37,37,Mohamed Salah,5,captain,1,0
2,3,39,39,Mohamed Salah,3,captain,1,7


In [6]:
# 1. who blanked, and who came on
import pandas as pd
from simulator import gw_slice, gw_actuals
for gw in [1,2,3]:
    els = log.loc[log.gw==gw, "elements"].iloc[0]
    a = gw_actuals(season, gw).set_index("element")
    s = season[(season.gw==gw) & (season.element.isin(els))][["element","name","position","e_points","e_minutes","minutes","actual_points"]]
    print(f"--- GW{gw} zero-minute players in squad ---")
    print(s[s.minutes==0].to_string(index=False))

# 2. the actual squad in GW1
print(season[(season.gw==1) & (season.element.isin(log.elements.iloc[0]))]
      [["name","position","team","e_points","e_minutes","minutes","actual_points"]]
      .sort_values("e_points", ascending=False).to_string(index=False))

--- GW1 zero-minute players in squad ---
 element            name position  e_points  e_minutes  minutes  actual_points
     251 Nicolas Jackson      FWD  2.863668  48.783658        0              0
     403  Joško Gvardiol      DEF  4.411132  72.181022        0              0
--- GW2 zero-minute players in squad ---
 element            name position  e_points  e_minutes  minutes  actual_points
     235     Cole Palmer      MID  5.957800  86.741725        0              0
     251 Nicolas Jackson      FWD  0.723122   9.728192        0              0
     403  Joško Gvardiol      DEF  1.216439  17.413538        0              0
--- GW3 zero-minute players in squad ---
 element              name position  e_points  e_minutes  minutes  actual_points
     251   Nicolas Jackson      FWD  0.509917   4.487826        0              0
     403    Joško Gvardiol      DEF  0.747982   9.706832        0              0
     610 Aaron Wan-Bissaka      DEF  3.483016  83.650263        0              0


In [10]:
state, log = simulate_season(season)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7
GW 4   41 pts  (total  164)  transfer     403->257  bank 3.6  bench  1
GW 5   27 pts  (total  191)  transfer     402->317  bank 5.0  bench  0
GW 6   36 pts  (total  227)  transfer     610->403  bank 3.5  bench  4
GW 7   41 pts  (total  268)  transfer      250->64  bank 1.0  bench  0


ValueError: transfer unaffordable: bank 10 + 50 - 63 = -3

In [9]:
state, log = simulate_season(season, gws=[1,2,3])

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))

from simulator import load_season, simulate_season
season = load_season()
state, log = simulate_season(season)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0


KeyboardInterrupt: 

In [1]:
from simulator import gw_slice, _adjusted_pool, decide_gameweek

ModuleNotFoundError: No module named 'simulator'

In [5]:
state30, _ = simulate_season(season, gws=list(range(1,31)), verbose=False)

gw = 31
pool = gw_slice(season, gw)
prices = dict(zip(pool["element"], pool["value"]))
adj = _adjusted_pool(pool, state30, prices)

print("pool size:", len(pool))
print("owned missing from pool:", [e for e in state30.elements if e not in set(pool.element)])
print("sell_value:", state30.sell_value(prices), "bank:", state30.bank,
      "power:", state30.budget(prices))
print("cost of keeping current 15 in adj prices:",
      int(adj.set_index('element').reindex(state30.elements)['value'].sum()))
print(adj.set_index('element').reindex(state30.elements)[['name','position','team','value']])

pool size: 664
owned missing from pool: [82, 1, 5, 267, 725]
sell_value: 964 bank: 1 power: 965
cost of keeping current 15 in adj prices: 643
                       name position         team  value
element                                                 
679          Mads Hermansen       GK     West Ham   42.0
593         Pape Matar Sarr      MID        Spurs   45.0
251         Nicolas Jackson      FWD      Chelsea   65.0
100           Junior Kroupi      FWD  Bournemouth   46.0
64            Ollie Watkins      FWD  Aston Villa   85.0
387      Dominik Szoboszlai      MID    Liverpool   68.0
82                      NaN      NaN          NaN    NaN
1                       NaN      NaN          NaN    NaN
374         Ibrahima Konaté      DEF    Liverpool   54.0
5                       NaN      NaN          NaN    NaN
267                     NaN      NaN          NaN    NaN
226         Trevoh Chalobah      DEF      Chelsea   55.0
381           Mohamed Salah      MID    Liverpool  140.0
343

In [6]:
missing = [82, 1, 5, 267, 725]
print(season[season.element.isin(missing)].groupby('element')['gw'].agg(['min','max','count']))
print("\npool sizes by gw:")
print(season.groupby('gw').size().to_string())

         min  max  count
element                 
1          1   38     37
5          1   38     37
82         1   38     36
267        1   38     37
725        4   38     34

pool sizes by gw:
gw
1     690
2     705
3     712
4     740
5     741
6     742
7     743
8     745
9     746
10    747
11    752
12    755
13    755
14    758
15    759
16    760
17    770
18    775
19    780
20    790
21    795
22    799
23    803
24    811
25    817
26    817
27    818
28    819
29    820
30    822
31    664
32    826
33    829
34    582
35    832
36    838
37    840
38    841


In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season
season = load_season()
state, log = simulate_season(season)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7
GW 4   41 pts  (total  164)  transfer     403->257  bank 3.6  bench  1
GW 5   27 pts  (total  191)  transfer     402->317  bank 5.0  bench  0
GW 6   36 pts  (total  227)  transfer     610->403  bank 3.5  bench  4
GW 7   41 pts  (total  268)  transfer      250->64  bank 1.0  bench  0
GW 8   52 pts  (total  320)  transfer     107->258  bank 1.0  bench  0
GW 9   59 pts  (total  379)  transfer     317->225  bank 0.0  bench  6
GW10   64 pts  (total  443)  transfer       17->20  bank 1.0  bench  4
GW11   53 pts  (total  496)  transfer       371->5  bank 0.1  bench  3
GW12   49 pts  (total  545)  transfer     225->436  bank 0.6  bench  2
GW13   39 pts  (total  584)  transfer       5->408  bank 1.5  bench  0
GW14   37 pts  (total  621)  transfer      20->387  bank 1.8  bench  2
GW15  

RuntimeError: GW31: no feasible squad found

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season
season = load_season()
state, log = simulate_season(season)


GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7
GW 4   41 pts  (total  164)  transfer     403->257  bank 3.6  bench  1
GW 5   27 pts  (total  191)  transfer     402->317  bank 5.0  bench  0
GW 6   36 pts  (total  227)  transfer     610->403  bank 3.5  bench  4
GW 7   41 pts  (total  268)  transfer      250->64  bank 1.0  bench  0
GW 8   52 pts  (total  320)  transfer     107->258  bank 1.0  bench  0
GW 9   59 pts  (total  379)  transfer     317->225  bank 0.0  bench  6
GW10   64 pts  (total  443)  transfer       17->20  bank 1.0  bench  4
GW11   53 pts  (total  496)  transfer       371->5  bank 0.1  bench  3
GW12   49 pts  (total  545)  transfer     225->436  bank 0.6  bench  2
GW13   39 pts  (total  584)  transfer       5->408  bank 1.5  bench  0
GW14   37 pts  (total  621)  transfer      20->387  bank 1.8  bench  2
GW15  

In [2]:
log.to_parquet(r"data\simulation_log.parquet", index=False)

In [3]:
from baselines import run_all_baselines
table, rand = run_all_baselines(season)

set and forget      : 1368
hindsight ceiling   : 2554
random (50 squads)  : mean 816, sd 225, range 361-1272


In [4]:
from baselines import set_and_forget, _reassign_roles
from simulator import gw_slice
import pandas as pd

total, log = set_and_forget(season)
print("set-and-forget per-gw:", round(total/38, 1))
print(log[["gw","points","captain_bonus","doubled_role","n_subs","bench_points"]].head(10))
print("\ntotal bench points left behind:", log.bench_points.sum())
print("captain never doubled in", (log.doubled_role=='none').sum(), "gameweeks")

set-and-forget per-gw: 36.0
   gw  points  captain_bonus doubled_role  n_subs  bench_points
0   1      47              8      captain       1             3
1   2      37              5      captain       1             0
2   3      35              3      captain       1             7
3   4      45              9      captain       1             1
4   5      23              5      captain       0             0
5   6      27              2      captain       0             0
6   7      27              2      captain       1             0
7   8      44              2      captain       0             0
8   9      43              6      captain       1             0
9  10      47             10      captain       0             0

total bench points left behind: 82
captain never doubled in 0 gameweeks


In [5]:
from bootstrap import compare_strategies, report, season_total_interval, sensitivity_to_block_length
from baselines import set_and_forget

saf_total, saf_log = set_and_forget(season)
r = compare_strategies(log["points"].values, saf_log["points"].values, label="set_and_forget")
report(r)

vs set_and_forget
  observed margin : +0 points
  95% interval    : [+0, +0]
  model loses in  : 100.0% of resampled seasons
  block length 5, 2000 resamples
  -> LUCK CANNOT BE RULED OUT -- the interval includes zero


In [6]:
print("model log rows:", len(log), " total:", log["points"].sum())
print("saf log rows:  ", len(saf_log), " total:", saf_log["points"].sum())
print("identical:", (log["points"].values == saf_log["points"].values).all())

model log rows: 38  total: 1368
saf log rows:   38  total: 1368
identical: True


In [7]:
from simulator import simulate_season
state, model_log = simulate_season(season, verbose=False)
print("model:", model_log["points"].sum(), " saf:", saf_log["points"].sum())

model: 1889  saf: 1368


In [8]:
model_log.to_parquet(r"data\simulation_log.parquet", index=False)
saf_log.to_parquet(r"data\baseline_saf_log.parquet", index=False)

r = compare_strategies(model_log["points"].values, saf_log["points"].values, label="set_and_forget")
report(r)

vs set_and_forget
  observed margin : +521 points
  95% interval    : [+343, +708]
  model loses in  : 0.0% of resampled seasons
  block length 5, 2000 resamples
  -> the margin is unlikely to be luck


In [9]:
print(sensitivity_to_block_length(model_log["points"].values, saf_log["points"].values).to_string(index=False))

 block_length  margin  ci_low  ci_high  excludes_zero  loses_pct
            1   521.0   368.0    664.0           True        0.0
            2   521.0   347.0    689.0           True        0.0
            4   521.0   347.0    693.0           True        0.0
            5   521.0   343.0    708.0           True        0.0
            6   521.0   337.0    705.0           True        0.0
            8   521.0   340.0    702.0           True        0.0
           10   521.0   347.0    694.0           True        0.0


In [10]:
from baselines import hindsight_set_and_forget
hind_total, hind_log = hindsight_set_and_forget(season)
hind_log.to_parquet(r"data\baseline_hindsight_log.parquet", index=False)

r2 = compare_strategies(model_log["points"].values, hind_log["points"].values, label="hindsight_ceiling")
report(r2)

vs hindsight_ceiling
  observed margin : -665 points
  95% interval    : [-958, -386]
  model loses in  : 100.0% of resampled seasons
  block length 5, 2000 resamples
  -> the margin is unlikely to be luck


In [11]:
import pulp
print([s for s in pulp.listSolvers(onlyAvailable=True)])

['PULP_CBC_CMD', 'HiGHS']


In [12]:
import time, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from optimize import load_gw_data, optimize_squad
import pulp

df = load_gw_data(gw=1)

t = time.time()
prob_cbc, _ = optimize_squad(df, mode="balanced")
cbc = time.time() - t

# same problem, HiGHS
t = time.time()
prob_h = pulp.LpProblem("x", pulp.LpMaximize)
# rebuild via optimize_squad but solve with HiGHS -- quick hack for timing
import optimize
orig = pulp.PULP_CBC_CMD
pulp.PULP_CBC_CMD = lambda **kw: pulp.HiGHS(msg=False)
prob_hi, _ = optimize_squad(df, mode="balanced")
pulp.PULP_CBC_CMD = orig
hi = time.time() - t

print(f"CBC   {cbc:.2f}s  obj {pulp.value(prob_cbc.objective):.4f}")
print(f"HiGHS {hi:.2f}s  obj {pulp.value(prob_hi.objective):.4f}")

CBC   1.24s  obj 61.2512
HiGHS 0.64s  obj 61.2512


In [1]:
import importlib, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, gw_slice
from transfer_mip import build_and_solve
import time

season = load_season()
pools = {gw: gw_slice(season, gw) for gw in [1,2,3,4,5,6]}

t = time.time()
status, plan = build_and_solve(pools, current_squad=[], purchase_prices={}, bank=0, free_transfers=1)
print(status, f"{time.time()-t:.1f}s")
for p in plan:
    print(p["gw"], "buys", len(p["buys"]), "sells", len(p["sells"]), "hits", p["hits"], f"decay {p['decay_weight']:.2f}")

Optimal 32.6s
1 buys 15 sells 0 hits 0 decay 1.00
2 buys 1 sells 1 hits 0 decay 0.85
3 buys 1 sells 1 hits 0 decay 0.72
4 buys 1 sells 1 hits 0 decay 0.61
5 buys 1 sells 1 hits 0 decay 0.52
6 buys 1 sells 1 hits 0 decay 0.44


In [2]:
# 1. Does it ever choose to take a hit if the payoff is there?
status2, plan2 = build_and_solve(pools, current_squad=[], purchase_prices={},
                                 bank=0, free_transfers=1, decay=1.0)
print("no decay:", [(p["gw"], len(p["buys"]), p["hits"]) for p in plan2])

# 2. How does solve time scale with horizon?
for H in [2, 4, 6]:
    pp = {gw: gw_slice(season, gw) for gw in range(1, 1+H)}
    t = time.time()
    s, _ = build_and_solve(pp, [], {}, 0, 1)
    print(f"H={H}: {time.time()-t:.1f}s")

no decay: [(1, 15, 0), (2, 1, 0), (3, 1, 0), (4, 1, 0), (5, 1, 0), (6, 1, 0)]
H=2: 6.5s
H=4: 11.7s
H=6: 33.1s


In [3]:
# give it 5 free transfers up front - if the projection is the constraint,
# it should now make several transfers in GW2
s, p = build_and_solve(pools, current_squad=[], purchase_prices={},
                       bank=0, free_transfers=5, decay=1.0)
print([(x["gw"], len(x["buys"]), x["hits"], x["free_transfers_assumed"]) for x in p])

[(1, 15, 0, 5), (2, 5, 0, 5), (3, 5, 0, 5), (4, 5, 0, 5), (5, 5, 0, 5), (6, 5, 0, 5)]


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))

from simulator import load_season, gw_slice
from transfer_mip import build_and_solve

season = load_season()
pools = {gw: gw_slice(season, gw) for gw in [1, 2, 3, 4, 5, 6]}

s, p = build_and_solve(pools, [], {}, 0, free_transfers=5, decay=1.0)
print("ft=5:", s, [(x["gw"], x["transfers_made"], x["hits"], x["free_transfers"]) for x in p])

s2, p2 = build_and_solve(pools, [], {}, 0, free_transfers=1, decay=0.85)
print("ft=1:", s2, [(x["gw"], x["transfers_made"], x["hits"], x["free_transfers"]) for x in p2])

ft=5: Optimal [(1, 15, 0, 5), (2, 1, 0, 5), (3, 2, 0, 5), (4, 3, 0, 4), (5, 1, 0, 2), (6, 2, 0, 2)]
ft=1: Optimal [(1, 15, 0, 1), (2, 2, 0, 2), (3, 1, 0, 1), (4, 1, 0, 1), (5, 1, 0, 1), (6, 1, 0, 1)]


In [2]:
s3, p3 = build_and_solve(pools, [], {}, 0, free_transfers=0, decay=1.0)
print("ft=0:", [(x["gw"], x["transfers_made"], x["hits"], x["free_transfers"]) for x in p3])

ft=0: [(1, 15, 0, 0), (2, 1, 0, 1), (3, 1, 0, 1), (4, 1, 0, 1), (5, 1, 0, 1), (6, 1, 0, 1)]


In [3]:
# GW2 alone: start with a squad, zero free transfers, so any transfer costs 4
_, p1 = build_and_solve({1: pools[1]}, [], {}, 0, free_transfers=1)
squad1 = p1[0]["squad"]
pp = {e: int(pools[1].set_index("element").loc[e, "value"]) for e in squad1}

for ft in [0, 1]:
    s, p = build_and_solve({2: pools[2]}, squad1, pp, bank=0, free_transfers=ft)
    print(f"ft={ft}:", s, "transfers", p[0]["transfers_made"], "hits", p[0]["hits"])

ft=0: Optimal transfers 0 hits 0
ft=1: Optimal transfers 1 hits 0


In [4]:
import pandas as pd
# make one unowned player absurdly good - worth far more than a -4
pool2 = pools[2].copy()
target = pool2[~pool2.element.isin(squad1)].iloc[0]
pool2.loc[pool2.element == target.element, "e_points"] = 50.0

s, p = build_and_solve({2: pool2}, squad1, pp, bank=0, free_transfers=0)
print(s, "transfers", p[0]["transfers_made"], "hits", p[0]["hits"])
print("bought the 50-pointer:", target.element in p[0]["buys"])

Optimal transfers 2 hits 2
bought the 50-pointer: True


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season

season = load_season()
gws = list(range(1, 9))

s1, l1 = simulate_season(season, gws=gws, policy="single", verbose=False)
print("single:", s1.total_points)

t = time.time()
s2, l2 = simulate_season(season, gws=gws, policy="mip", horizon=6, verbose=False)
print(f"mip H=6: {s2.total_points}  ({time.time()-t:.0f}s)")
print(l2[["gw","points","n_transfers","hit","effective_horizon"]].to_string(index=False))

single: 320
mip H=6: 452  (39s)
 gw  points  n_transfers  hit  effective_horizon
  1      47            0    0                  1
  2      37            5   12                  6
  3      52            1    0                  6
  4      83            1    0                  5
  5      42            1    0                  4
  6      73            1    0                  3
  7      54            1    0                  2
  8      64            1    0                  1


In [2]:
import pandas as pd
from scipy.stats import spearmanr

wf = pd.read_parquet(r"data\walkforward_h6_2526.parquet")
v = wf.dropna(subset=["actual_points","e_points"]).copy()
v["actual_points"] = pd.to_numeric(v["actual_points"], errors="coerce")

bands = {
    "All rows":        lambda d: d,
    "Played (>0)":     lambda d: d[d.minutes > 0],
    "Started (60+)":   lambda d: d[d.minutes >= 60],
}

print(f"{'step':<6}" + "".join(f"{b:>18}" for b in bands))
for step in sorted(v.horizon_step.unique()):
    s = v[v.horizon_step == step]
    row = f"{step:<6}"
    for f in bands.values():
        d = f(s)
        row += f"{spearmanr(d.e_points, d.actual_points).correlation:>10.3f} (n={len(d)//1000}k)"
    print(row)

step            All rows       Played (>0)     Started (60+)
0          0.715 (n=29k)     0.337 (n=11k)     0.099 (n=7k)
1          0.682 (n=28k)     0.294 (n=10k)     0.087 (n=7k)
2          0.661 (n=27k)     0.276 (n=10k)     0.097 (n=7k)
3          0.641 (n=26k)     0.249 (n=10k)     0.080 (n=6k)
4          0.629 (n=25k)     0.236 (n=9k)     0.076 (n=6k)
5          0.617 (n=24k)     0.221 (n=9k)     0.068 (n=6k)


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season

season_h = load_season(horizon_aware=True)
gws = list(range(1, 9))

s1, l1 = simulate_season(season_h, gws=gws, policy="single", verbose=False)
print("single:", s1.total_points)

t = time.time()
s2, l2 = simulate_season(season_h, gws=gws, policy="mip", horizon=6, verbose=False)
print(f"mip H=6: {s2.total_points}  ({time.time()-t:.0f}s)")

single: 320


PulpError: Cannot multiply variables with NaN/inf values

In [2]:
import pandas as pd
season_h = load_season(horizon_aware=True)

print("NaN e_points:", season_h["e_points"].isna().sum(), "of", len(season_h))
print("\nby horizon_step:")
print(season_h.groupby("horizon_step")["e_points"].apply(lambda s: s.isna().sum()))
print("\nsample rows:")
print(season_h[season_h["e_points"].isna()].head(5)[
    ["element","name","cutoff","gw","horizon_step","e_minutes","p_start","npxg90"]])

NaN e_points: 3701 of 165401

by horizon_step:
horizon_step
0      0
1    560
2    694
3    782
4    912
5    753
Name: e_points, dtype: int64

sample rows:
      element                   name  cutoff  gw  horizon_step  e_minutes  \
4140      691  Dominic Calvert-Lewin       1   2             1        NaN   
4141      691  Dominic Calvert-Lewin       1   3             2        NaN   
4142      691  Dominic Calvert-Lewin       1   4             3        NaN   
4143      691  Dominic Calvert-Lewin       1   5             4        NaN   
4144      691  Dominic Calvert-Lewin       1   6             5        NaN   

      p_start    npxg90  
4140      NaN  0.465205  
4141      NaN  0.465205  
4142      NaN  0.465205  
4143      NaN  0.465205  
4144      NaN  0.465205  


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "squad"))

from simulator import load_season, simulate_season

season = load_season()
state, log = simulate_season(season, policy="mip",
                             gws=list(range(1, 11)),
                             wildcard_gw=5)

print(log[["gw", "n_transfers", "hit", "wildcard", "points"]].to_string(index=False))
print(f"\n10-GW total: {state.total_points}")

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer 235->26, 251->430, 371->5, 403->295, 413->387  bank 0.8  bench  7
GW 3   52 pts  (total  136)  transfer     610->476  bank 0.2  bench 11
GW 4   83 pts  (total  219)  transfer     402->257  bank 1.2  bench  9
GW 5   40 pts  (total  259)  transfer 250->691, 593->427  bank 1.3  bench 20  WILDCARD
GW 6   79 pts  (total  338)  transfer 107->72, 679->1  bank 0.4  bench 18
GW 7   60 pts  (total  398)  transfer            -  bank 0.4  bench 10
GW 8   69 pts  (total  467)  transfer 72->225, 295->348  bank 0.3  bench 31
GW 9   64 pts  (total  531)  transfer     257->684  bank 0.4  bench 16
GW10   74 pts  (total  605)  transfer     348->261  bank 0.0  bench 11
 gw  n_transfers  hit  wildcard  points
  1            0    0     False      47
  2            5   12     False      37
  3            1    0     False      52
  4            1    0     False      83
  5            2    0      True  

In [2]:
import importlib, transfer_mip, simulator
importlib.reload(transfer_mip); importlib.reload(simulator)

from simulator import load_season, simulate_season

for wc in [None, 5]:
    st, lg = simulate_season(season, policy="mip", gws=list(range(1, 11)),
                             wildcard_gw=wc, verbose=False)
    row = lg[lg["gw"] == 5].iloc[0]
    print(f"wildcard_gw={str(wc):>4}  GW5 transfers={row['n_transfers']}  "
          f"hit={row['hit']}  10gw total={st.total_points}")

wildcard_gw=None  GW5 transfers=1  hit=0  10gw total=590
wildcard_gw=   5  GW5 transfers=2  hit=0  10gw total=605


In [3]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season()

t0 = time.time()
st, lg = simulate_season(season, policy="mip", horizon=3, decay=0.45,
                         verbose=True)
print(f"\n{st.total_points} pts in {time.time()-t0:.0f}s")

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   42 pts  (total   89)  transfer 251->97, 413->82  bank 0.8  bench  2
GW 3   46 pts  (total  135)  transfer     235->119  bank 3.2  bench  9
GW 4   64 pts  (total  199)  transfer     610->257  bank 2.6  bench  1
GW 5   36 pts  (total  235)  transfer     402->505  bank 3.0  bench  0
GW 6   51 pts  (total  286)  transfer 250->430, 381->427  bank 3.7  bench 18
GW 7   78 pts  (total  364)  transfer       107->5  bank 2.5  bench  4
GW 8   68 pts  (total  432)  transfer     593->387  bank 1.0  bench 24
GW 9   64 pts  (total  496)  transfer     403->225  bank 1.4  bench 13
GW10   75 pts  (total  571)  transfer      427->20  bank 0.1  bench  3
GW11   49 pts  (total  620)  transfer     371->226  bank 0.7  bench  4
GW12   61 pts  (total  681)  transfer     225->373  bank 0.2  bench  2
GW13   52 pts  (total  733)  transfer       5->407  bank 1.2  bench  6
GW14   75 pts  (total  808)  transfer      20->414  bank 0.0  bench

In [4]:
import time
import pandas as pd
from tqdm import tqdm

H, DECAY = 3, 0.45

print("baseline running (~2 min)...", flush=True)
t0 = time.time()
base_state, _ = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                                verbose=False)
BASE = base_state.total_points
print(f"baseline (no wildcard): {BASE} pts  [{time.time()-t0:.0f}s]\n", flush=True)

rows = []
for k in tqdm(range(1, 39, 3), desc="wildcard sweep"):
    st, lg = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                             wildcard_gw=k, verbose=False)
    wc = lg[lg["gw"] == k].iloc[0]
    rows.append({"wildcard_gw": k, "total": st.total_points,
                 "gain": st.total_points - BASE,
                 "transfers_that_gw": int(wc["n_transfers"])})
    print(f"  GW{k:2d}: {st.total_points:5d}  gain {st.total_points-BASE:+5d}  "
          f"({int(wc['n_transfers'])} transfers)", flush=True)

sweep = pd.DataFrame(rows)
sweep.to_parquet("data/wildcard_sweep_coarse.parquet", index=False)
print(f"\nBest: GW{int(sweep.loc[sweep['gain'].idxmax(),'wildcard_gw'])}  "
      f"{sweep['gain'].max():+d}")
print(f"Spread: {sweep['gain'].min():+d} to {sweep['gain'].max():+d}")

baseline running (~2 min)...
baseline (no wildcard): 2083 pts  [129s]



wildcard sweep:   0%|          | 0/13 [00:00<?, ?it/s]

  GW 1:  2083  gain    +0  (0 transfers)


wildcard sweep:   8%|▊         | 1/13 [02:07<25:25, 127.11s/it]

  GW 4:  2082  gain    -1  (2 transfers)


...

  GW34:  2101  gain   +18  (2 transfers)


wildcard sweep:  92%|█████████▏| 12/13 [24:35<02:06, 126.90s/it]

  GW37:  2071  gain   -12  (2 transfers)


wildcard sweep: 100%|██████████| 13/13 [26:42<00:00, 123.25s/it]


Best: GW13  +59
Spread: -50 to +59


In [5]:
import importlib, transfer_mip
importlib.reload(transfer_mip)

for wc in [None, 0]:
    st, lg = simulate_season(season, policy="mip", horizon=3, decay=0.45,
                             gws=list(range(1, 15)),
                             wildcard_gw=13 if wc == 0 else None, verbose=False)
    r = lg[lg["gw"] == 13].iloc[0]
    print(f"wildcard={'yes' if wc==0 else 'no ':>3}  transfers={r['n_transfers']}  "
          f"hit={r['hit']}  bank={r['bank']}")

wildcard=no   transfers=1  hit=0  bank=7
wildcard=yes  transfers=2  hit=0  bank=2


In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season()
st, lg = simulate_season(season, policy="mip", horizon=3, decay=0.45, verbose=True)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   42 pts  (total   89)  transfer 251->97, 413->82  bank 0.8  bench  2
GW 3   46 pts  (total  135)  transfer     235->119  bank 3.2  bench  9
GW 4   64 pts  (total  199)  transfer     610->257  bank 2.6  bench  1
GW 5   36 pts  (total  235)  transfer     402->505  bank 3.0  bench  0
GW 6   51 pts  (total  286)  transfer 250->430, 381->427  bank 3.7  bench 18
GW 7   78 pts  (total  364)  transfer       107->5  bank 2.5  bench  4
GW 8   68 pts  (total  432)  transfer     593->387  bank 1.0  bench 24
GW 9   64 pts  (total  496)  transfer     403->225  bank 1.4  bench 13
GW10   75 pts  (total  571)  transfer      427->20  bank 0.1  bench  3
GW11   49 pts  (total  620)  transfer     371->226  bank 0.7  bench  4
GW12   61 pts  (total  681)  transfer     225->373  bank 0.2  bench  2
GW13   52 pts  (total  733)  transfer       5->407  bank 1.2  bench  6
GW14   75 pts  (total  808)  transfer      20->414  bank 0.0  bench

In [3]:
from pathlib import Path
import datetime, pandas as pd

for f in ["walkforward_2526.parquet", "walkforward_h6_2526.parquet"]:
    p = Path("data") / f
    if p.exists():
        t = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        print(f"{f:32s}  {p.stat().st_size/1e6:6.1f} MB   built {t:%Y-%m-%d %H:%M}")
    else:
        print(f"{f:32s}  MISSING")

print("\nWhich file does load_season use?")
import inspect, simulator
print(inspect.getsource(simulator.load_season)[:900])

walkforward_2526.parquet             4.5 MB   built 2026-08-07 22:52
walkforward_h6_2526.parquet         15.4 MB   built 2026-08-10 15:48

Which file does load_season use?
def load_season(walkforward_path=None, history_path=HISTORY_PATH,
                horizon_aware=False):
    """Load as-of predictions, realized outcomes, and prices for every gameweek.

    horizon_aware=False (default) loads the original file: one prediction per
    player-gameweek, made with that gameweek's own cutoff. Correct for a
    single-gameweek policy, which never looks past the gameweek it is playing.

    horizon_aware=True loads the horizon file, which carries a `cutoff` column.
    The multi-gameweek MIP NEEDS this. Handing it the original file would be a
    leak that is easy to miss: gameweek k+3's row there was produced with cutoff
    k+3, so its rolling features were built from gameweeks k+1 and k+2 -- matches
    that have not been played when the planner is standing at gameweek k.

    Returns a 

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)
st, lg = simulate_season(season, policy="mip", horizon=3, decay=0.45, verbose=True)

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   41 pts  (total   88)  transfer 403->5, 413->17  bank 0.3  bench  2
GW 3   39 pts  (total  127)  transfer     235->119  bank 2.7  bench  9
GW 4   43 pts  (total  170)  transfer      17->267  bank 4.1  bench  1
GW 5   23 pts  (total  193)  transfer 251->64, 610->257  bank 1.0  bench  0
GW 6   60 pts  (total  253)  transfer 250->430, 381->427, 402->403  bank 1.8  bench 27
GW 7   72 pts  (total  325)  transfer       679->1  bank 0.7  bench  2
GW 8   84 pts  (total  409)  transfer     107->225  bank 0.3  bench 27
GW 9   78 pts  (total  487)  transfer     267->387  bank 0.3  bench 11
GW10   76 pts  (total  563)  transfer     371->261  bank 1.6  bench 13
GW11   50 pts  (total  613)  transfer      427->20  bank 0.2  bench 10
GW12   49 pts  (total  662)  transfer     225->258  bank 0.7  bench 10
GW13   38 pts  (total  

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)
st, lg = simulate_season(season, policy="mip", horizon=3, decay=0.45, verbose=True)

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   44 pts  (total   91)  transfer 403->541, 413->16  bank 0.3  bench  2
GW 3   39 pts  (total  130)  transfer     235->119  bank 2.7  bench  7
GW 4   40 pts  (total  170)  transfer      16->267  bank 6.0  bench  1
GW 5   28 pts  (total  198)  transfer 251->430, 402->683  bank 0.2  bench  0
GW 6   71 pts  (total  269)  transfer 267->427, 610->295  bank 0.7  bench 12
GW 7   42 pts  (total  311)  transfer     541->532  bank 0.2  bench  0
GW 8   72 pts  (total  383)  transfer     107->694  bank 1.2  bench 25
GW 9   59 pts  (total  442)  transfer       371->5  bank 0.6  bench  6
GW10   69 pts  (total  511)  transfer     683->261  bank 0.1  bench  6
GW11   56 pts  (total  567)  transfer      119->20  bank 1.4  bench  4
GW12   51 pts  (total  618)  transfer     427->387  bank 0.2  bench  3
GW13   34 pts  (total  652)  t

In [2]:
import pandas as pd
wf = pd.read_parquet("data/walkforward_h6_2526.parquet")

# 1. Do predictions collapse at any point in the season?
print(wf[wf.horizon_step == 0].groupby("gw")["e_points"].mean().round(2).to_string())

# 2. Sanity: are the step-0 rows identical to what assembly produces?
print("\nrows:", len(wf), " cutoffs:", wf.cutoff.nunique(),
      " steps:", sorted(wf.horizon_step.unique()))
print(wf.groupby("horizon_step")["e_points"].mean().round(3))

gw
1     1.45
2     1.54
3     1.49
4     1.47
5     1.44
6     1.45
7     1.46
8     1.44
9     1.45
10    1.46
11    1.42
12    1.44
13    1.43
14    1.44
15    1.44
16    1.43
17    1.41
18    1.40
19    1.40
20    1.40
21    1.39
22    1.38
23    1.37
24    1.37
25    1.36
26    1.38
27    1.37
28    1.35
29    1.32
30    1.33
31    1.29
32    1.31
33    1.37
34    1.32
35    1.34
36    1.34
37    1.31
38    1.31

rows: 165401  cutoffs: 38  steps: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
horizon_step
0    1.395
1    1.407
2    1.413
3    1.416
4    1.416
5    1.419
Name: e_points, dtype: float64


In [1]:
import sys, time
from pathlib import Path
from tqdm import tqdm
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)

rows = []
grid = [0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6]
for d in tqdm(grid, desc="decay sweep (H=3)"):
    t0 = time.time()
    st, lg = simulate_season(season, policy="mip", horizon=3, decay=d,
                             verbose=False)
    rows.append({"decay": d, "total": st.total_points})
    print(f"  decay {d:.2f}: {st.total_points:5d}  ({time.time()-t0:.0f}s)",
          flush=True)

sweep = pd.DataFrame(rows)
print("\n" + sweep.to_string(index=False))
best = sweep.loc[sweep["total"].idxmax()]
print(f"\nBest: decay {best['decay']} -> {int(best['total'])}")
print("Reference: single-transfer v1 = 1889, set-and-forget = 1368")

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff


decay sweep (H=3):   0%|          | 0/7 [00:00<?, ?it/s]

  decay 0.25:  1901  (134s)


decay sweep (H=3):  14%|█▍        | 1/7 [02:13<13:22, 133.72s/it]

  decay 0.30:  1973  (111s)


...

  decay 0.50:  1887  (168s)


decay sweep (H=3):  86%|████████▌ | 6/7 [13:18<02:21, 141.55s/it]

  decay 0.60:  1955  (174s)


decay sweep (H=3): 100%|██████████| 7/7 [16:12<00:00, 138.87s/it]


 decay  total
  0.25   1901
  0.30   1973
  0.35   1907
  0.40   1880
  0.45   1886
  0.50   1887
  0.60   1955

Best: decay 0.3 -> 1973
Reference: single-transfer v1 = 1889, set-and-forget = 1368


In [2]:
import numpy as np, pandas as pd
from tqdm import tqdm
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season
from bootstrap import compare_strategies, report, season_total_interval, sensitivity_to_block_length
from baselines import set_and_forget

season = load_season(horizon_aware=True)

logs = {}
for d in tqdm([0.3, 0.5], desc="re-running the two configs"):
    st, lg = simulate_season(season, policy="mip", horizon=3, decay=d, verbose=False)
    logs[d] = lg
    print(f"  decay {d}: {st.total_points}", flush=True)

# single-transfer reference and the naive baseline
st1, lg1 = simulate_season(season, policy="single", verbose=False)
saf_total, saf_log = set_and_forget(season)
print(f"  single-transfer: {st1.total_points}   set-and-forget: {saf_total}")

print("\n--- Is decay 0.3 really better than 0.5? ---")
report(compare_strategies(logs[0.3]["points"].values,
                          logs[0.5]["points"].values, label="decay 0.5"))

print("\n--- Does the MIP beat the single-transfer search? ---")
report(compare_strategies(logs[0.3]["points"].values,
                          lg1["points"].values, label="single-transfer v1"))

print("\n--- Does the MIP beat set-and-forget? ---")
report(compare_strategies(logs[0.3]["points"].values,
                          saf_log["points"].values, label="set-and-forget"))

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff


re-running the two configs:   0%|          | 0/2 [00:00<?, ?it/s]

  decay 0.3: 1973


re-running the two configs:  50%|█████     | 1/2 [02:13<02:13, 133.34s/it]

  decay 0.5: 1887


re-running the two configs: 100%|██████████| 2/2 [04:43<00:00, 141.70s/it]


  single-transfer: 1889   set-and-forget: 1368

--- Is decay 0.3 really better than 0.5? ---
vs decay 0.5
  observed margin : +86 points
  95% interval    : [-70, +233]
  model loses in  : 14.8% of resampled seasons
  block length 5, 2000 resamples
  -> LUCK CANNOT BE RULED OUT -- the interval includes zero

--- Does the MIP beat the single-transfer search? ---
vs single-transfer v1
  observed margin : +84 points
  95% interval    : [-24, +188]
  model loses in  : 6.2% of resampled seasons
  block length 5, 2000 resamples
  -> LUCK CANNOT BE RULED OUT -- the interval includes zero

--- Does the MIP beat set-and-forget? ---
vs set-and-forget
  observed margin : +605 points
  95% interval    : [+409, +790]
  model loses in  : 0.0% of resampled seasons
  block length 5, 2000 resamples
  -> the margin is unlikely to be luck


In [1]:
import sys, time
from pathlib import Path
from tqdm import tqdm
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

H, DECAY = 3, 0.3

season = load_season(horizon_aware=True)
assert "cutoff" in season.columns, "wrong file -- MIP would read k+1 as-of-k+1"
assert (season["cutoff"] <= season["gw"]).all(), "cutoff after target gameweek"
print(f"OK: horizon file, {len(season):,} rows\n", flush=True)

t0 = time.time()

with tqdm(total=39, desc="wildcard sweep", unit="season") as bar:
    st, _ = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                            verbose=False)
    BASE = st.total_points
    bar.update(1); bar.write(f"baseline (no chips): {BASE}\n")

    rows = []
    for k in range(1, 20):
        st, lg = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                                 wildcard_gws=k, verbose=False)
        rows.append({"wc1": k, "total": st.total_points,
                     "gain": st.total_points - BASE,
                     "transfers": int(lg[lg.gw == k].iloc[0]["n_transfers"])})
        bar.update(1)
        bar.write(f"  WC1 GW{k:2d}: {st.total_points}  {st.total_points-BASE:+5d}")

    h1 = pd.DataFrame(rows)
    h1.to_parquet("data/wc_half1.parquet", index=False)
    BEST1 = int(h1.loc[h1["gain"].idxmax(), "wc1"])
    bar.write(f"\n  -> best first half: GW{BEST1} {h1['gain'].max():+d}  "
              f"(spread {h1['gain'].min():+d} to {h1['gain'].max():+d})\n")

    rows2 = []
    for k in range(20, 39):
        st, lg = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                                 wildcard_gws=[BEST1, k], verbose=False)
        rows2.append({"wc1": BEST1, "wc2": k, "total": st.total_points,
                      "gain_vs_base": st.total_points - BASE,
                      "transfers": int(lg[lg.gw == k].iloc[0]["n_transfers"])})
        bar.update(1)
        bar.write(f"  WC1@{BEST1} + WC2 GW{k:2d}: {st.total_points}  "
                  f"{st.total_points-BASE:+5d}")

h2 = pd.DataFrame(rows2)
h2.to_parquet("data/wc_half2.parquet", index=False)
BEST2 = int(h2.loc[h2["gain_vs_base"].idxmax(), "wc2"])

print(f"\nElapsed: {(time.time()-t0)/60:.1f} min")
print(f"baseline            : {BASE}")
print(f"one wildcard  GW{BEST1:2d}  : {int(h1['total'].max())}  "
      f"{h1['gain'].max():+d}   spread {h1['gain'].min():+d}..{h1['gain'].max():+d}")
print(f"two wildcards GW{BEST1},{BEST2}: {int(h2['total'].max())}  "
      f"{h2['gain_vs_base'].max():+d}   spread "
      f"{h2['gain_vs_base'].min():+d}..{h2['gain_vs_base'].max():+d}")
print("\nHindsight upper bound; greedy on the first half. Read the SPREAD first --")
print("run-to-run noise on this config is roughly +/-100.")

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
OK: horizon file, 161,700 rows



wildcard sweep:   3%|▎         | 1/39 [01:57<1:14:31, 117.67s/season]

baseline (no chips): 1984



wildcard sweep:   5%|▌         | 2/39 [03:50<1:10:44, 114.72s/season]

  WC1 GW 1: 1984     +0


...

  WC1 GW 6: 1928    -56


wildcard sweep:  21%|██        | 8/39 [32:01<2:15:11, 261.65s/season]

  WC1 GW 7: 1890    -94


wildcard sweep:  21%|██        | 8/39 [33:38<2:10:20, 252.28s/season]


KeyboardInterrupt: 

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)

for i in range(3):
    st, _ = simulate_season(season, policy="mip", horizon=3, decay=0.3,
                            verbose=False)
    print(f"run {i+1}: {st.total_points}", flush=True)

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
run 1: 1984


KeyboardInterrupt: 

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)

for k in [2, 7]:
    _, a = simulate_season(season, policy="mip", horizon=3, decay=0.3,
                           gws=list(range(1, k+1)), verbose=False)
    _, b = simulate_season(season, policy="mip", horizon=3, decay=0.3,
                           gws=list(range(1, k+1)), wildcard_gws=k, verbose=False)
    ra, rb = a.iloc[-1], b.iloc[-1]
    print(f"GW{k}: base {ra['points']:3d} ({ra['n_transfers']}tr, hit {ra['hit']}, "
          f"bank {ra['bank']})   wc {rb['points']:3d} ({rb['n_transfers']}tr, "
          f"hit {rb['hit']}, bank {rb['bank']})   flag={rb['wildcard']}")

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
GW2: base  41 (2tr, hit 0, bank 3)   wc  60 (11tr, hit 0, bank 2)   flag=True
GW7: base  44 (1tr, hit 0, bank 1)   wc  74 (10tr, hit 0, bank 2)   flag=True


In [2]:
for i in range(3):
    st, _ = simulate_season(season, policy="mip", horizon=3, decay=0.3,
                            verbose=False)
    print(f"run {i+1}: {st.total_points}", flush=True)

run 1: 1984
run 2: 1984


KeyboardInterrupt: 

In [3]:
import hashlib
from pathlib import Path
p = Path("data/walkforward_h6_2526.parquet")
print("modified:", p.stat().st_mtime)
print("size:", p.stat().st_size)
print("md5:", hashlib.md5(p.read_bytes()).hexdigest())

modified: 1786424412.0024292
size: 15465319
md5: c90b97d490731ce99bed90a68300d1d3


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)
st, _ = simulate_season(season, policy="mip", horizon=3, decay=0.3, verbose=False)
print(st.total_points)

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
1973


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)
st, _ = simulate_season(season, policy="mip", horizon=3, decay=0.3, verbose=False)
print(st.total_points)

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
1984


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

season = load_season(horizon_aware=True)
for i in range(3):
    st, _ = simulate_season(season, policy="mip", horizon=3, decay=0.3, verbose=False)
    print(f"run {i+1}: {st.total_points}", flush=True)

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff
run 1: 1984
run 2: 1984
run 3: 1984


In [1]:
import sys, time
from pathlib import Path
from tqdm import tqdm
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "squad"))
from simulator import load_season, simulate_season

H, DECAY, W = 3, 0.3, 4          # W = local window length in gameweeks

season = load_season(horizon_aware=True)
assert "cutoff" in season.columns, "wrong file -- MIP would read k+1 as-of-k+1"
assert (season["cutoff"] <= season["gw"]).all(), "cutoff after target gameweek"

t0 = time.time()

with tqdm(total=39, desc="wildcard sweep", unit="season") as bar:
    st, base_log = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                                   verbose=False)
    BASE = st.total_points
    bp = base_log.set_index("gw")["points"]
    bar.update(1); bar.write(f"baseline: {BASE}\n")

    rows = []
    for k in range(1, 39):
        st, lg = simulate_season(season, policy="mip", horizon=H, decay=DECAY,
                                 wildcard_gws=k, verbose=False)
        p = lg.set_index("gw")["points"]
        win = [g for g in range(k, min(k + W, 39))]
        rows.append({
            "gw": k,
            "local_gain": int(p.loc[win].sum() - bp.loc[win].sum()),
            "season_gain": int(st.total_points - BASE),
            "transfers": int(lg[lg.gw == k].iloc[0]["n_transfers"]),
            "season_hits": int(lg["hit"].sum()),
        })
        bar.update(1)
        r = rows[-1]
        bar.write(f"  GW{k:2d}: local {r['local_gain']:+4d}   "
                  f"season {r['season_gain']:+5d}   ({r['transfers']} tr)")

sw = pd.DataFrame(rows)
sw.to_parquet("data/wc_sweep_fixed.parquet", index=False)

print(f"\nElapsed {(time.time()-t0)/60:.1f} min   baseline {BASE}   "
      f"baseline hits {int(base_log['hit'].sum())}")
print(f"\nLOCAL (GW k..k+{W-1}) -- what the chip mechanically buys")
print(f"  mean {sw.local_gain.mean():+.1f}   median {sw.local_gain.median():+.0f}"
      f"   range {sw.local_gain.min():+d}..{sw.local_gain.max():+d}")
print(f"  best GW{int(sw.loc[sw.local_gain.idxmax(),'gw'])} "
      f"{sw.local_gain.max():+d}")
print(f"\nSEASON -- local effect plus 30+ weeks of path divergence")
print(f"  mean {sw.season_gain.mean():+.1f}   range "
      f"{sw.season_gain.min():+d}..{sw.season_gain.max():+d}")
print(f"\nMean transfers on the chip week: {sw.transfers.mean():.1f} "
      f"(should be ~10, not ~2)")

[load_season] dropped 3701 rows (2.2%) with no prediction -- players not yet visible at their cutoff


wildcard sweep:   3%|▎         | 1/39 [01:48<1:08:40, 108.44s/season]

baseline: 1984



wildcard sweep:   5%|▌         | 2/39 [03:41<1:08:33, 111.19s/season]

  GW 1: local   +0   season    +0   (0 tr)


...

  GW36: local  -11   season   -11   (8 tr)


wildcard sweep:  97%|█████████▋| 38/39 [6:20:18<12:41, 761.85s/season] 

  GW37: local   +9   season    +9   (8 tr)


wildcard sweep: 100%|██████████| 39/39 [6:22:13<00:00, 588.03s/season]

  GW38: local  +24   season   +24   (8 tr)

Elapsed 382.2 min   baseline 1984   baseline hits 20

LOCAL (GW k..k+3) -- what the chip mechanically buys
  mean +12.3   median +7   range -33..+82
  best GW3 +82

SEASON -- local effect plus 30+ weeks of path divergence
  mean -26.2   range -159..+95

Mean transfers on the chip week: 10.6 (should be ~10, not ~2)


In [2]:
print("lg" in dir())

True


In [3]:
best = lg.nlargest(5, "captain_bonus")[["gw", "captain", "captain_bonus"]]
print(best.to_string(index=False))
print(f"\nSeason mean:  +{lg['captain_bonus'].mean():.1f}")
print(f"Best GW1-19:  +{lg[lg.gw<=19]['captain_bonus'].max()}")
print(f"Best GW20-38: +{lg[lg.gw>19]['captain_bonus'].max()}")

 gw            captain  captain_bonus
  6     Erling Haaland             16
 17     Erling Haaland             16
 14     Erling Haaland             14
 16        Bukayo Saka             11
 35 Keane Lewis-Potter             11

Season mean:  +5.5
Best GW1-19:  +16
Best GW20-38: +11


In [4]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
KEYWORDS = ["status", "news", "chance_of_playing", "as_of", "available"]

print("=== parquet/csv files and any availability columns ===\n")
files = sorted(list(ROOT.glob("data/**/*.parquet")) + list(ROOT.glob("data/**/*.csv")))

for f in files:
    try:
        if f.suffix == ".parquet":
            cols = list(pd.read_parquet(f, engine="pyarrow").head(0).columns)
        else:
            cols = list(pd.read_csv(f, nrows=0).columns)
    except Exception as e:
        print(f"{f.relative_to(ROOT)}  -- unreadable: {e}")
        continue

    hits = [c for c in cols if any(k in c.lower() for k in KEYWORDS)]
    mark = "  <<< " + ", ".join(hits) if hits else ""
    print(f"{str(f.relative_to(ROOT)):55s} {len(cols):4d} cols{mark}")

=== parquet/csv files and any availability columns ===

data\baseline_hindsight_log.parquet                        8 cols
data\baseline_saf_log.parquet                              8 cols
data\history\2016-17\merged_gw.csv  -- unreadable: 'utf-8' codec can't decode byte 0xe9 in position 1358: invalid continuation byte
data\history\2016-17\merged_gw_with_position.csv          58 cols
data\history\2016-17\players_raw.csv                      57 cols  <<< chance_of_playing_next_round, chance_of_playing_this_round, news, status
data\history\2017-18\merged_gw.csv  -- unreadable: 'utf-8' codec can't decode byte 0xe9 in position 1547: invalid continuation byte
data\history\2017-18\merged_gw_with_position.csv          58 cols
data\history\2017-18\players_raw.csv                      58 cols  <<< chance_of_playing_next_round, chance_of_playing_this_round, news, news_added, status
data\history\2018-19\merged_gw.csv  -- unreadable: 'utf-8' codec can't decode byte 0xe9 in position 1799: invalid co

In [5]:
import pandas as pd
from pathlib import Path

for season in ["2019-20", "2022-23", "2024-25", "2025-26"]:
    p = Path("data/history") / season / "players_raw.csv"
    if not p.exists():
        print(f"{season}: players_raw.csv MISSING")
        continue
    df = pd.read_csv(p)
    cols = [c for c in df.columns if c in
            ("status", "news", "chance_of_playing_next_round", "gw", "round")]
    print(f"\n{season}: {len(df)} rows, {df.columns.size} cols")
    print(f"  availability cols: {cols}")
    if "status" in df:
        print(f"  status values: {df['status'].value_counts().to_dict()}")
        print(f"  rows per player id: {len(df) / df['id'].nunique():.2f}"
              "   (1.0 = single snapshot, no history)")


2019-20: 666 rows, 61 cols
  availability cols: ['chance_of_playing_next_round', 'news', 'status']
  status values: {'a': 474, 'u': 64, 'n': 61, 'i': 53, 'd': 11, 's': 3}
  rows per player id: 1.00   (1.0 = single snapshot, no history)
2022-23: players_raw.csv MISSING
2024-25: players_raw.csv MISSING
2025-26: players_raw.csv MISSING


In [6]:
import pandas as pd

df = pd.read_parquet("data/history/all_seasons_fixed.parquet")
print("all_seasons_fixed columns:")
print(sorted(df.columns.tolist()))

print("\n--- merged_gw column check ---")
for s in ["2020-21", "2022-23", "2024-25", "2025-26"]:
    cols = pd.read_csv(f"data/history/{s}/merged_gw.csv", nrows=0,
                       encoding="latin-1").columns.tolist()
    hits = [c for c in cols
            if any(k in c.lower() for k in
                   ["status", "news", "chance", "avail", "flag"])]
    print(f"{s}: {len(cols)} cols   availability: {hits or 'none'}")

all_seasons_fixed columns:
['GW', 'assists', 'attempted_passes', 'big_chances_created', 'big_chances_missed', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'completed_passes', 'creativity', 'defensive_contribution', 'dribbles', 'ea_index', 'element', 'errors_leading_to_goal', 'errors_leading_to_goal_attempt', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'fouls', 'goals_conceded', 'goals_scored', 'ict_index', 'id', 'influence', 'key_passes', 'kickoff_time', 'kickoff_time_formatted', 'loaned_in', 'loaned_out', 'minutes', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win', 'modified', 'name', 'offside', 'open_play_crosses', 'opponent_team', 'own_goals', 'penalties_conceded', 'penalties_missed', 'penalties_saved', 'position', 'recoveries', 'red_cards', 'round', 'saves', 'season', 'selected', 'starts', 'tackled', 'tackles', 'target_missed', 'team'

In [8]:
from pathlib import Path
for p in Path("data").rglob("*core_insights*"):
    print(p)
    

data\data description\FPL_core_insights_gw_reference.docx
data\data description\FPL_core_insights_matchstats_reference.docx
data\history\core_insights
data\history\core_insights_gameweek_stats.parquet
data\history\core_insights_gameweek_stats_fixed.parquet
data\history\core_insights_matchstats.parquet


In [9]:
import pandas as pd

for f in ["core_insights_gameweek_stats.parquet",
          "core_insights_gameweek_stats_fixed.parquet"]:
    ci = pd.read_parquet(f"data/history/{f}")
    cols = [c for c in ["status", "chance_of_playing_this_round",
                        "chance_of_playing_next_round", "news"]
            if c in ci.columns]
    print(f"\n=== {f} ===  {len(ci):,} rows")
    print(f"availability cols present: {cols}")
    if not cols:
        continue
    g = ci.groupby("gw").apply(lambda x: pd.Series({
        "n": len(x),
        "status_nonA": (x["status"] != "a").sum() if "status" in x else 0,
        "cop_next_notnull": x["chance_of_playing_next_round"].notna().sum()
                            if "chance_of_playing_next_round" in x else 0,
        "news_nonempty": (x["news"].fillna("") != "").sum() if "news" in x else 0,
    }), include_groups=False)
    print(g.to_string())


=== core_insights_gameweek_stats.parquet ===  29,978 rows
availability cols present: ['status', 'chance_of_playing_this_round', 'chance_of_playing_next_round', 'news']
      n  status_nonA  cop_next_notnull  news_nonempty
gw                                                   
1   759          254               462            254
2   752          140               752            285
3   752          148               752            287
4   752          206               752            290
5   752          201               752            284
6   752          231               752            285
7   752          235               752            286
8   752          232               752            276
9   752          232               752            271
10  752          232               748            232
11  752          247               427            247
12  755          240               436            240
13  755          242               439            242
14  758          252 

In [10]:
import pandas as pd

ci = pd.read_parquet("data/history/core_insights_gameweek_stats.parquet")

a = ci[ci.gw == 1].set_index("id")[["status", "news", "chance_of_playing_next_round"]]
b = ci[ci.gw == 15].set_index("id")[["status", "news", "chance_of_playing_next_round"]]
common = a.index.intersection(b.index)
print(f"players in both: {len(common)} (GW1 {len(a)}, GW15 {len(b)})")

for col in a.columns:
    same = (a.loc[common, col].fillna("~") == b.loc[common, col].fillna("~")).mean()
    print(f"  {col:32s} identical in {same:.1%} of players")

# For contrast: adjacent weeks should NOT be identical either, but should be closer
c = ci[ci.gw == 16].set_index("id")[["status"]]
common2 = b.index.intersection(c.index)
print(f"\nGW15 vs GW16 status identical: "
      f"{(b.loc[common2,'status'] == c.loc[common2,'status']).mean():.1%}")

# And a real early-season contrast
d = ci[ci.gw == 2].set_index("id")[["status"]]
common3 = a.index.intersection(d.index)
print(f"GW1  vs GW2  status identical: "
      f"{(a.loc[common3,'status'] == d.loc[common3,'status']).mean():.1%}")

players in both: 759 (GW1 759, GW15 759)
  status                           identical in 100.0% of players
  news                             identical in 100.0% of players
  chance_of_playing_next_round     identical in 100.0% of players

GW15 vs GW16 status identical: 88.4%
GW1  vs GW2  status identical: 71.8%


In [11]:
import pandas as pd
import numpy as np

ci = pd.read_parquet("data/history/core_insights_gameweek_stats.parquet")
cols = ["status", "news", "chance_of_playing_next_round"]
gws = sorted(ci.gw.unique())

snap = {g: ci[ci.gw == g].set_index("id")[cols] for g in gws}
M = pd.DataFrame(index=gws, columns=gws, dtype=float)
for i in gws:
    for j in gws:
        common = snap[i].index.intersection(snap[j].index)
        if len(common) < 50:
            continue
        M.loc[i, j] = np.mean([
            (snap[i].loc[common, c].fillna("~") ==
             snap[j].loc[common, c].fillna("~")).mean() for c in cols])

# Any NON-adjacent pair that is 100% identical is a backfill
print("Suspicious pairs (identical, more than 1 gameweek apart):")
for i in gws:
    for j in gws:
        if j > i + 1 and M.loc[i, j] > 0.999:
            print(f"  GW{i} == GW{j}")

Suspicious pairs (identical, more than 1 gameweek apart):
  GW1 == GW15


In [12]:
import pandas as pd

ci = pd.read_parquet("data/history/core_insights_gameweek_stats.parquet")
col = "chance_of_playing_next_round"

print("value distribution by gameweek (excluding nulls):\n")
for g in [2, 5, 9, 10, 11, 15, 20, 30, 38]:
    x = ci[ci.gw == g][col]
    vc = x.value_counts().sort_index().to_dict()
    print(f"GW{g:2d}: {x.notna().sum():4d} non-null / {len(x)}   {vc}")

print("\n--- cross-check against status ---")
print("If the field is genuine, 100 should pair with status 'a',")
print("and low values with 'i'/'d'.\n")
for g in [5, 11, 20]:
    x = ci[ci.gw == g]
    print(f"GW{g}:")
    print(pd.crosstab(x["status"], x[col].fillna("NULL")).to_string())
    print()

value distribution by gameweek (excluding nulls):

GW 2:  752 non-null / 752   {0.0: 679, 25.0: 6, 50.0: 5, 75.0: 15, 100.0: 47}
GW 5:  752 non-null / 752   {0.0: 629, 25.0: 10, 50.0: 11, 75.0: 10, 100.0: 92}
GW 9:  752 non-null / 752   {0.0: 570, 25.0: 3, 50.0: 11, 75.0: 11, 100.0: 157}
GW10:  748 non-null / 752   {0.0: 555, 25.0: 6, 50.0: 6, 75.0: 10, 100.0: 171}
GW11:  427 non-null / 752   {0.0: 218, 25.0: 3, 50.0: 11, 75.0: 15, 100.0: 180}
GW15:  462 non-null / 759   {0.0: 224, 25.0: 11, 50.0: 1, 75.0: 18, 100.0: 208}
GW20:  512 non-null / 790   {0.0: 246, 25.0: 22, 50.0: 6, 75.0: 18, 100.0: 220}
GW30:  605 non-null / 822   {0.0: 277, 25.0: 2, 50.0: 3, 75.0: 17, 100.0: 306}
GW38:  630 non-null / 841   {0.0: 267, 25.0: 3, 50.0: 4, 75.0: 4, 100.0: 352}

--- cross-check against status ---
If the field is genuine, 100 should pair with status 'a',
and low values with 'i'/'d'.

GW5:
chance_of_playing_next_round  0.0    25.0   50.0   75.0   100.0
status                                    

In [13]:
from pathlib import Path
root = Path("fplcache/cache")
years = sorted(p.name for p in root.iterdir() if p.is_dir())
print("years:", years)
for y in years:
    months = sorted(p.name for p in (root/y).iterdir() if p.is_dir())
    print(f"  {y}: {months}")

years: ['2021', '2022', '2023', '2024', '2025', '2026']
  2021: ['10', '11', '12', '4', '5', '6', '7', '8', '9']
  2022: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2023: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2024: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2025: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2026: ['1', '2', '3', '4', '5', '6', '7', '8']


In [14]:
from pathlib import Path
root = Path("fplcache/cache")
years = sorted(p.name for p in root.iterdir() if p.is_dir())
print("years:", years)
for y in years:
    months = sorted(p.name for p in (root/y).iterdir() if p.is_dir())
    print(f"  {y}: {months}")

years: ['2021', '2022', '2023', '2024', '2025', '2026']
  2021: ['10', '11', '12', '4', '5', '6', '7', '8', '9']
  2022: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2023: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2024: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2025: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']
  2026: ['1', '2', '3', '4', '5', '6', '7', '8']


In [15]:
from pathlib import Path
import lzma, json

root = Path("fplcache/cache")

# how many snapshots per month across the season
for y, m in [("2025", "8"), ("2025", "9"), ("2025", "12"), ("2026", "3"), ("2026", "5")]:
    p = root / y / m
    n = sum(1 for _ in p.rglob("*.json.xz"))
    days = len(list(p.iterdir()))
    print(f"{y}-{m:>2}: {n:3d} snapshots across {days} days")

# open one and check the fields are intact
f = sorted((root / "2025" / "12").rglob("*.json.xz"))[0]
data = json.loads(lzma.open(f).read())
e = data["elements"][0]
print(f"\nsample file: {f}")
print(f"players: {len(data['elements'])}")
for k in ["id", "web_name", "status", "news", "news_added",
          "chance_of_playing_this_round", "chance_of_playing_next_round"]:
    print(f"  {k:32s} {e.get(k)!r}")

2025- 8: 124 snapshots across 31 days
2025- 9: 119 snapshots across 30 days
2025-12: 123 snapshots across 31 days
2026- 3: 124 snapshots across 31 days
2026- 5: 123 snapshots across 31 days

sample file: fplcache\cache\2025\12\1\0228.json.xz
players: 755
  id                               1
  web_name                         'Raya'
  status                           'a'
  news                             ''
  news_added                       None
  chance_of_playing_this_round     None
  chance_of_playing_next_round     None


In [ ]:
from pathlib import Path
import lzma, json

f = sorted((Path("fplcache/cache")/"2025"/"12"/"1").rglob("*.json.xz"))[0]
data = json.loads(lzma.open(f).read())

hurt = [e for e in data["elements"] if e["status"] != "a"]
print(f"{len(hurt)} of {len(data['elements'])} not available\n")
for e in hurt[:8]:
    print(f"{e['web_name']:18s} {e['status']}  "
          f"cop={e['chance_of_playing_next_round']}  "
          f"added={e['news_added']}  {e['news'][:55]}")

241 of 755 not available

Hein               u  cop=0  added=2025-08-26T13:44:02.357864Z  Has joined Werder Bremen on loan for the rest of the se
Gabriel            i  cop=0  added=2025-11-17T12:00:07.436586Z  Thigh injury - Unknown return date
Saliba             d  cop=75  added=2025-11-30T22:30:08.957329Z  Knock - 75% chance of playing
Kiwior             u  cop=0  added=2025-09-02T13:37:25.929594Z  has joined Porto on loan for the rest of the season.
Kacurri            u  cop=0  added=2025-09-22T21:14:30.857843Z  has joined Morecambe on loan for the rest of the season
Trossard           d  cop=75  added=2025-11-27T10:00:06.582480Z  Knock - 75% chance of playing
Fábio Vieira       u  cop=0  added=2025-09-02T13:38:06.128321Z  has joined Hamburger on loan for the rest of the season
Kabia              u  cop=0  added=2025-09-02T13:24:41.013511Z  has joined Shrewsbury Town on loan for the rest of the 


: 